# LeetCode 1017 — Convert to Base -2 bằng Python

Link đề: https://leetcode.com/problems/convert-to-base-2/ · Độ khó: **Medium** · Chủ đề: Math (toán), Bit Manipulation (thao tác bit)

Skeleton LeetCode:

```python
class Solution:
    def baseNeg2(self, n: int) -> str:
        ...
```

Ghi chú: cho số nguyên `n` (`0 <= n <= 10^9`), trả về **chuỗi nhị phân** biểu diễn `n` trong **cơ số `-2`**. Trong cơ số `-2`, chữ số thứ `i` (tính từ phải, bắt đầu từ `0`) có **trọng số `(-2)^i`**: `1, -2, 4, -8, 16, ...`, nên `"110" = (-2)^2 + (-2)^1 = 2`. Chuỗi trả về **không có số `0` đứng đầu** (trừ chính `"0"`). Biểu diễn trong cơ số âm là **duy nhất** nên ta so sánh **đúng chuỗi** (không phải bài nhiều đáp án). Thuật toán là phép chia liên tiếp cho `-2` giống đổi cơ số thường, nhưng phải chăm sóc dấu của thương và số dư; ngoài ra còn có một mẹo bit rất gọn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu cơ số âm: trọng số `(-2)^i` xen kẽ dấu, vì sao mọi số nguyên (kể cả âm) có biểu diễn **duy nhất** chỉ dùng chữ số `0` và `1`.
2. Nắm các công cụ Python cần thiết: `n & 1`, `n >> 1`, `divmod` và dấu của số dư khi chia cho số **âm**, `''.join(reversed(...))`, `bin(...)`, và phép XOR với mặt nạ.
3. Hiểu nguyên lý chia liên tiếp: chữ số thấp nhất là `r = n mod 2` (`0` hoặc `1`, **không âm**), phần còn lại `q = (n - r) / (-2)` là số nguyên; lặp cho đến khi `n = 0`, rồi **đảo** các chữ số thu được.
4. Đi qua 4 cách: sinh mọi chuỗi nhị phân ngắn và tra bảng (brute force, oracle), chia liên tiếp bằng `n & 1` (cách nộp chính), `divmod` với số dư âm phải hiệu chỉnh, và mẹo bit `(n + M) ^ M` với mặt nạ `M = 0xAAAA...`.
5. Tránh các lỗi kinh điển: dùng `n //= -2` (chia sàn) mà không trừ số dư trước, quên trường hợp `n = 0`, quên đảo chuỗi, hiệu chỉnh số dư âm mà quên cộng `1` vào thương, và trả chuỗi có số `0` đứng đầu.

## 2. Đề bài & ví dụ

Cho số nguyên `n`, trả về **chuỗi nhị phân** biểu diễn `n` trong **cơ số `-2`**.

Lưu ý chuỗi trả về **không được có số `0` đứng đầu** trừ khi chuỗi là `"0"`.

**Ví dụ 1:** `n = 2` -> `"110"`. Giải thích: `(-2)^2 + (-2)^1 = 4 - 2 = 2`.

**Ví dụ 2:** `n = 3` -> `"111"`. Giải thích: `(-2)^2 + (-2)^1 + (-2)^0 = 4 - 2 + 1 = 3`.

**Ví dụ 3:** `n = 4` -> `"100"`. Giải thích: `(-2)^2 = 4`.

**Ràng buộc:**

- `0 <= n <= 10^9`

In [1]:
from typing import Dict, List
import random, time

def decode(s: str) -> int:
    """Giá trị của chuỗi nhị phân trong cơ số -2 (dùng để kiểm tra độc lập)."""
    return sum(int(c) * (-2) ** i for i, c in enumerate(reversed(s)))

examples = [(2, "110"), (3, "111"), (4, "100")]
for n, expected in examples:
    print(f"n={n} -> {expected!r} | giải mã {expected!r} = {decode(expected)}")
    assert decode(expected) == n

n=2 -> '110' | giải mã '110' = 2
n=3 -> '111' | giải mã '111' = 3
n=4 -> '100' | giải mã '100' = 4


## 3. Kiến thức nền

### 3.1. Cơ số âm

Trong cơ số `-2`, chuỗi `d_k ... d_1 d_0` (mỗi `d_i` là `0` hoặc `1`) có giá trị `sum d_i * (-2)^i`. Trọng số: `1, -2, 4, -8, 16, -32, ...`. Các vị trí **lẻ** (`i = 1, 3, ...`) có trọng số **âm**, vị trí **chẵn** có trọng số **dương**. Vì thế mọi số nguyên (cả âm lẫn dương) biểu diễn được **mà không cần dấu trừ**, và biểu diễn (không có `0` đứng đầu) là **duy nhất**.

In [2]:
print("trọng số:", [(-2) ** i for i in range(8)])
for s in ("0", "1", "10", "11", "110", "111", "100", "1101"):
    print(f"{s:5} -> {decode(s)}")

trọng số: [1, -2, 4, -8, 16, -32, 64, -128]
0     -> 0
1     -> 1
10    -> -2
11    -> -1
110   -> 2
111   -> 3
100   -> 4
1101  -> -3


### 3.2. Chia liên tiếp để đổi cơ số

Đổi cơ số thường: `n = b * q + r` với `0 <= r < b`; `r` là chữ số thấp nhất, rồi tiếp tục với `q`. Với cơ số `-2`: `n = (-2) * q + r` với `r` thuộc `{0, 1}` (số dư **không âm**). Suy ra `r = n mod 2` (tính chẵn lẻ của `n`: `n & 1`, **đúng cả với `n` âm** trong Python), và `q = (n - r) / (-2) = -((n - r) // 2)`. Lặp cho đến khi `n = 0`; các chữ số được tạo từ **thấp lên cao**, nên cuối cùng phải **đảo** chuỗi.

In [3]:
n = 6
digits = []
while n:
    r = n & 1
    digits.append(r)
    q = -((n - r) // 2)
    print(f"n={n:3}: r={r}, q = -(({n} - {r}) // 2) = {q}")
    n = q
print("chữ số từ thấp lên cao:", digits, "-> chuỗi:", "".join(str(d) for d in reversed(digits)))
assert "".join(str(d) for d in reversed(digits)) == "11010" and decode("11010") == 6

n=  6: r=0, q = -((6 - 0) // 2) = -3
n= -3: r=1, q = -((-3 - 1) // 2) = 2
n=  2: r=0, q = -((2 - 0) // 2) = -1
n= -1: r=1, q = -((-1 - 1) // 2) = 1
n=  1: r=1, q = -((1 - 1) // 2) = 0
chữ số từ thấp lên cao: [0, 1, 0, 1, 1] -> chuỗi: 11010


### 3.3. Chia cho số âm trong Python: `//`, `%` và `divmod`

Python **chia sàn** và số dư có **dấu theo số chia**: `divmod(3, -2) == (-2, -1)`: thương `-2`, số dư `-1` (vì `3 = (-2) * (-2) + (-1)`). Số dư `-1` không nằm trong `{0, 1}`, nên cần **hiệu chỉnh**: cộng `2` vào số dư (được `1`) và cộng `1` vào thương (vì `(-2)*(q + 1) + (r + 2) = (-2)*q + r`... kiểm tra: `-2 * (-1) + 1 = 3`). Vì vậy **không** thể dùng thẳng `n //= -2`.

In [4]:
for n in (2, 3, 6, 7):
    q, r = divmod(n, -2)
    print(f"divmod({n}, -2) = ({q}, {r}) | n // -2 = {n // -2} (chia sàn)", end=" | ")
    if r < 0:
        q, r = q + 1, r + 2
    print(f"hiệu chỉnh -> thương {q}, chữ số {r}, kiểm tra: (-2)*{q} + {r} = {(-2) * q + r}")

divmod(2, -2) = (-1, 0) | n // -2 = -1 (chia sàn) | hiệu chỉnh -> thương -1, chữ số 0, kiểm tra: (-2)*-1 + 0 = 2
divmod(3, -2) = (-2, -1) | n // -2 = -2 (chia sàn) | hiệu chỉnh -> thương -1, chữ số 1, kiểm tra: (-2)*-1 + 1 = 3
divmod(6, -2) = (-3, 0) | n // -2 = -3 (chia sàn) | hiệu chỉnh -> thương -3, chữ số 0, kiểm tra: (-2)*-3 + 0 = 6
divmod(7, -2) = (-4, -1) | n // -2 = -4 (chia sàn) | hiệu chỉnh -> thương -3, chữ số 1, kiểm tra: (-2)*-3 + 1 = 7


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ `n = 3`:

- `n = 3`: lẻ, chữ số `1`; `q = -((3 - 1) // 2) = -1`.
- `n = -1`: lẻ (`-1 & 1 = 1`), chữ số `1`; `q = -((-1 - 1) // 2) = -(-1) = 1`.
- `n = 1`: chữ số `1`; `q = -((1 - 1) // 2) = 0`. Dừng.

Chữ số theo thứ tự tạo ra: `1, 1, 1`; đảo: `"111"`. Kiểm tra: `4 - 2 + 1 = 3` ✓.

Ví dụ `n = 2`: `n = 2`: chẵn, chữ số `0`, `q = -(2 // 2) = -1`; `n = -1`: chữ số `1`, `q = 1`; `n = 1`: chữ số `1`, `q = 0`. Chữ số: `0, 1, 1` -> đảo `"110"` ✓.

## 5. Cách 1 — Brute Force: sinh mọi chuỗi ngắn và tra bảng

**Ý tưởng.** Biết rằng biểu diễn là duy nhất, ta **liệt kê mọi chuỗi nhị phân** độ dài `<= L` (không bắt đầu bằng `0` trừ `"0"`), tính giá trị của chúng theo định nghĩa và lưu `giá trị -> chuỗi` vào từ điển. Mọi `n` trong phạm vi bảng được tra ngay. Không dùng chia hay mẹo bit nên đây là **oracle độc lập** (và còn cho kiểm chứng tính duy nhất: hai chuỗi khác nhau không thể cùng giá trị).

In [5]:
def build_table(max_len: int) -> Dict[int, str]:
    table: Dict[int, str] = {0: "0"}
    for length in range(1, max_len + 1):
        for bits in range(1 << (length - 1), 1 << length):            # chuỗi bắt đầu bằng 1
            s = format(bits, "b")
            v = decode(s)
            assert v not in table, ("hai chuỗi cùng giá trị", v, s, table[v])   # tính duy nhất
            table[v] = s
    return table

table = build_table(16)
print("số giá trị phân biệt (độ dài <= 16):", len(table), "| ví dụ:", {n: table[n] for n in range(0, 9)})
print("các số không âm 0..N đều có mặt với N =", next(k for k in range(0, 10 ** 6) if k not in table) - 1, "(= 1 + 4 + 16 + ... + 4^7)")

def base_neg2_brute(n: int) -> str:
    return table[n]

for n, expected in examples:
    assert base_neg2_brute(n) == expected
print("oracle khớp ví dụ")

số giá trị phân biệt (độ dài <= 16): 65536 | ví dụ: {0: '0', 1: '1', 2: '110', 3: '111', 4: '100', 5: '101', 6: '11010', 7: '11011', 8: '11000'}
các số không âm 0..N đều có mặt với N = 21845 (= 1 + 4 + 16 + ... + 4^7)
oracle khớp ví dụ


**Độ phức tạp và nút thắt.** Số chuỗi độ dài `<= L` là `2^L`: bảng với `L = 16` có `65536` mục và chỉ phủ `n <= 21845`; để phủ `n = 10^9` cần `L ~ 31` (`2^31` chuỗi): không thể. Đo thử rồi ngoại suy:

In [6]:
for L in (10, 14, 16, 18):
    t0 = time.perf_counter()
    build_table(L)
    dt = time.perf_counter() - t0
    print(f"L={L:2}: {dt * 1000:9.1f} ms, bảng có {2 ** L} mục")
print("để phủ n = 10^9 cần L >= 31: ~ 2^31 chuỗi, không thể")

L=10:       3.0 ms, bảng có 1024 mục
L=14:      60.1 ms, bảng có 16384 mục


L=16:     187.0 ms, bảng có 65536 mục


L=18:     771.7 ms, bảng có 262144 mục
để phủ n = 10^9 cần L >= 31: ~ 2^31 chuỗi, không thể


Nút thắt: ta không cần liệt kê; với mỗi `n` ta **tính trực tiếp** từng chữ số từ thấp lên cao bằng phép chia liên tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Chữ số thấp nhất quyết định bởi tính chẵn lẻ; phần còn lại là một bài toán nhỏ hơn.**

Vì `(-2)^i` chẵn với mọi `i >= 1`, chữ số thấp nhất `d_0` phải có cùng tính chẵn lẻ với `n`: `d_0 = n & 1`. Bỏ chữ số đó đi, giá trị còn lại là `n - d_0` chia hết cho `2`, và phần còn lại biểu diễn `(n - d_0) / (-2)` (chia cho `-2` vì mọi trọng số còn lại đều chia hết cho `-2`). Lặp lại. Biên độ `|n|` giảm khoảng một nửa mỗi bước nên có `O(log n)` bước.

| Cách | Ý tưởng |
|---|---|
| 2 | `r = n & 1`; `n = -((n - r) // 2)` (nộp) |
| 3 | `divmod(n, -2)`; nếu số dư `< 0` thì cộng `2` vào số dư và `1` vào thương |
| 4 | Mẹo bit: `(n + M) ^ M` với `M = 0xAAAAAAAA...` (bit `1` ở các vị trí lẻ) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — `n & 1` và `n = -((n - r) // 2)` (cách nộp chính)

**Giải thích từng đoạn.**

- `n == 0` thì trả `"0"` (vòng lặp sẽ không chạy nên phải xử lý riêng).
- `r = n & 1`: chữ số thấp nhất (đúng cho cả `n` âm trong Python).
- `n = -((n - r) // 2)`: `n - r` chẵn nên chia `2` chính xác (không dính chia sàn), rồi **đổi dấu** (chia cho `-2`).
- Chữ số được thêm theo thứ tự **từ thấp lên cao** nên đảo trước khi ghép.

In [7]:
def base_neg2_loop(n: int) -> str:
    if n == 0:
        return "0"
    digits: List[str] = []
    while n:
        r = n & 1
        digits.append(str(r))
        n = -((n - r) // 2)
    return "".join(reversed(digits))

for n, expected in examples:
    got = base_neg2_loop(n)
    print(f"n={n}: {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

n=2: '110' OK
n=3: '111' OK
n=4: '100' OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def baseNeg2(self, n: int) -> str:
        if n == 0:
            return "0"
        digits = []
        while n:
            r = n & 1                      # chữ số thấp nhất = tính chẵn lẻ của n
            digits.append(str(r))
            n = -((n - r) // 2)            # (n - r) chia hết cho 2; chia cho -2
        return "".join(reversed(digits))   # chữ số tạo từ thấp lên cao nên đảo lại

sol_loop = Solution().baseNeg2
for n, expected in examples:
    assert sol_loop(n) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(log n)` (khoảng `log2(n) + 2` bước), bộ nhớ `O(log n)` cho các chữ số.

---

### 7.2. Cách 3 — `divmod` với số dư âm phải hiệu chỉnh

**Ý tưởng khác.** Dùng `q, r = divmod(n, -2)`: thương theo chia sàn và số dư `r` thuộc `{0, -1}` (số dư cùng dấu số chia). Nếu `r == -1`: hiệu chỉnh `r = 1` và `q += 1` (vì `-2 * (q + 1) + 1 = -2 * q - 1`).

**Giải thích từng đoạn.**

- `n, r = divmod(n, -2)` cập nhật `n` thành thương.
- `if r < 0: r += 2; n += 1`: đưa chữ số về `{0, 1}` và bù vào thương.
- Phần còn lại giống Cách 2.

In [9]:
def base_neg2_divmod(n: int) -> str:
    if n == 0:
        return "0"
    digits: List[str] = []
    while n:
        n, r = divmod(n, -2)
        if r < 0:                          # r == -1: đưa về chữ số 1 và bù vào thương
            r += 2
            n += 1
        digits.append(str(r))
    return "".join(reversed(digits))

for n, expected in examples:
    got = base_neg2_divmod(n)
    print(f"n={n}: {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

n=2: '110' OK
n=3: '111' OK
n=4: '100' OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def baseNeg2(self, n: int) -> str:
        if n == 0:
            return "0"
        digits = []
        while n:
            n, r = divmod(n, -2)           # r thuộc {0, -1}
            if r < 0:
                r += 2                     # chữ số phải là 0 hoặc 1
                n += 1                     # bù vào thương
            digits.append(str(r))
        return "".join(reversed(digits))

sol_divmod = Solution().baseNeg2
for n, expected in examples:
    assert sol_divmod(n) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** như Cách 2: thời gian `O(log n)`, bộ nhớ `O(log n)`.

---

### 7.3. Cách 4 — Mẹo bit với mặt nạ `0xAAAA...`

**Ý tưởng khác.** Với mặt nạ `M` có bit `1` ở mọi vị trí **lẻ** (`M = ...1010 1010`, tức `0xAAAAAAAA...`), công thức `(n + M) ^ M` cho **trực tiếp** chuỗi bit của `n` trong cơ số `-2`. Lý do: xét các bit tại vị trí lẻ, có trọng số `-2^i`; ta viết `n = P - N` với `P` là tổng các bit chẵn và `N` là tổng các bit lẻ nhân `2^i`; cộng `M` rồi XOR `M` "lật" đúng các bit ở vị trí lẻ để chuyển trọng số `+2^i` thành `-2^i`.

**Giải thích từng đoạn.**

- `M = 0xAAAAAAAAAAAAAAAA` (64 bit, đủ cho `n <= 10^9`, cả cho số âm nhỏ).
- `v = (n + M) ^ M`, rồi `bin(v)[2:]` ra chuỗi không có `0` đứng đầu (`bin(0)[2:] = "0"` nên `n = 0` tự đúng).
- Chứng minh chặt chẽ ở mục 10 (và kiểm chứng bằng máy trên mọi `n <= 10^5` cộng mẫu ngẫu nhiên).

In [11]:
MASK = 0xAAAAAAAAAAAAAAAA

def base_neg2_mask(n: int) -> str:
    return bin((n + MASK) ^ MASK)[2:]

for n, expected in examples:
    got = base_neg2_mask(n)
    print(f"n={n}: {got!r}", "OK" if got == expected else "SAI")
    assert got == expected
print("n = 0 ->", repr(base_neg2_mask(0)))

n=2: '110' OK
n=3: '111' OK
n=4: '100' OK
n = 0 -> '0'


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def baseNeg2(self, n: int) -> str:
        M = 0xAAAAAAAAAAAAAAAA             # bit 1 ở mọi vị trí lẻ
        return bin((n + M) ^ M)[2:]       # bin(0)[2:] = "0" nên n = 0 đúng

sol_mask = Solution().baseNeg2
for n, expected in examples:
    assert sol_mask(n) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(1)` (vài phép số học trên số nguyên 64 bit) cộng `O(log n)` để in chuỗi; bộ nhớ `O(log n)`.

## 8. TRACE từng bước

### 8.1. Cách 2 cho `n = 10`

In `n`, chữ số và thương ở mỗi bước, cùng giá trị riêng phần.

In [13]:
def trace_loop(n):
    start = n
    digits = []
    step = 0
    while n:
        r = n & 1
        q = -((n - r) // 2)
        print(f"bước {step}: n={n:4} chẵn lẻ r={r}, q = -(({n} - {r}) // 2) = {q}, trọng số (-2)^{step} = {(-2) ** step}")
        digits.append(str(r))
        n = q
        step += 1
    s = "".join(reversed(digits)) or "0"
    print(f"chuỗi = {s!r}, giải mã = {decode(s)} (phải bằng {start})")
    return s

assert trace_loop(10) == "11110"

bước 0: n=  10 chẵn lẻ r=0, q = -((10 - 0) // 2) = -5, trọng số (-2)^0 = 1
bước 1: n=  -5 chẵn lẻ r=1, q = -((-5 - 1) // 2) = 3, trọng số (-2)^1 = -2
bước 2: n=   3 chẵn lẻ r=1, q = -((3 - 1) // 2) = -1, trọng số (-2)^2 = 4
bước 3: n=  -1 chẵn lẻ r=1, q = -((-1 - 1) // 2) = 1, trọng số (-2)^3 = -8
bước 4: n=   1 chẵn lẻ r=1, q = -((1 - 1) // 2) = 0, trọng số (-2)^4 = 16
chuỗi = '11110', giải mã = 10 (phải bằng 10)


### 8.2. Cách 3: khi nào cần hiệu chỉnh số dư

In `divmod(n, -2)` và hiệu chỉnh cho `n = 7`.

In [14]:
def trace_divmod(n):
    digits = []
    while n:
        q, r = divmod(n, -2)
        note = ""
        if r < 0:
            q, r, note = q + 1, r + 2, "  (số dư -1 -> hiệu chỉnh: +2 cho chữ số, +1 cho thương)"
        print(f"n={n:3}: chữ số {r}, thương {q}{note}")
        digits.append(str(r))
        n = q
    return "".join(reversed(digits))

assert trace_divmod(7) == "11011" and decode("11011") == 7

n=  7: chữ số 1, thương -3  (số dư -1 -> hiệu chỉnh: +2 cho chữ số, +1 cho thương)
n= -3: chữ số 1, thương 2  (số dư -1 -> hiệu chỉnh: +2 cho chữ số, +1 cho thương)
n=  2: chữ số 0, thương -1
n= -1: chữ số 1, thương 1  (số dư -1 -> hiệu chỉnh: +2 cho chữ số, +1 cho thương)
n=  1: chữ số 1, thương 0  (số dư -1 -> hiệu chỉnh: +2 cho chữ số, +1 cho thương)


### 8.3. Cách 4: các bước của mẹo bit cho `n = 3`

In `M` (rút gọn 8 bit thấp), `n + M`, kết quả XOR.

In [15]:
n = 3
M8 = 0xAA
print(f"n      = {n:08b}")
print(f"M (8b) = {M8:08b}")
print(f"n + M  = {(n + M8) & 0xFF:08b}  ({n + M8})")
print(f"(n+M)^M= {((n + M8) ^ M8) & 0xFF:08b}  -> chuỗi {bin(((n + M8) ^ M8) & 0xFF)[2:]!r}, giải mã = {decode(bin(((n + M8) ^ M8) & 0xFF)[2:])}")
assert bin((n + MASK) ^ MASK)[2:] == "111"

n      = 00000011
M (8b) = 10101010
n + M  = 10101101  (173)
(n+M)^M= 00000111  -> chuỗi '111', giải mã = 3


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (`n & 1` và `n = -((n - r) // 2)`): dễ hiểu, `O(log n)`, không cần hiệu chỉnh.

```python
class Solution:
    def baseNeg2(self, n: int) -> str:
        if n == 0:
            return "0"
        digits = []
        while n:
            r = n & 1                      # chữ số thấp nhất = tính chẵn lẻ của n
            digits.append(str(r))
            n = -((n - r) // 2)            # (n - r) chia hết cho 2; chia cho -2
        return "".join(reversed(digits))   # chữ số tạo từ thấp lên cao nên đảo lại
```

In [16]:
class Solution:
    def baseNeg2(self, n: int) -> str:
        if n == 0:
            return "0"
        digits = []
        while n:
            r = n & 1                      # chữ số thấp nhất = tính chẵn lẻ của n
            digits.append(str(r))
            n = -((n - r) // 2)            # (n - r) chia hết cho 2; chia cho -2
        return "".join(reversed(digits))   # chữ số tạo từ thấp lên cao nên đảo lại

sol = Solution()
for n, expected in examples:
    got = sol.baseNeg2(n)
    print(f"n={n}: {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

n=2: '110' OK
n=3: '111' OK
n=4: '100' OK


## 10. Vì sao đúng

**Tồn tại và duy nhất.** Ta chứng minh quy nạp rằng thuật toán chia liên tiếp cho ra một chuỗi biểu diễn `n`. Với `n` bất kỳ, đặt `r = n mod 2` thuộc `{0, 1}` và `q = (r - n) / 2 = -((n - r) / 2)`. Khi đó `n = r + (-2) * q`. Nếu `q` có biểu diễn `d_k ... d_0` (giá trị `sum d_i (-2)^i = q`) thì `n = r + (-2) * sum d_i (-2)^i = r + sum d_i (-2)^(i+1)`: chuỗi `d_k ... d_0 r` biểu diễn `n`.

**Dừng.** `|q| = |n - r| / 2 <= (|n| + 1) / 2`. Với `|n| >= 2`: `|q| <= (|n| + 1)/2 < |n|`; với `n = 1`: `q = 0`; với `n = -1`: `r = 1`, `q = 1`, rồi `1 -> 0`. Vậy quy trình dừng sau `O(log |n|)` bước.

**Không có `0` đứng đầu.** Chữ số cuối cùng sinh ra (cao nhất) là chữ số của bước mà `n != 0`: nếu `n != 0` và bước sau cho `q = 0` thì `n - r = 0` nên `n = r = 1` (chữ số `1`). Vậy chữ số cao nhất luôn là `1`.

**Duy nhất.** Hai biểu diễn khác nhau của cùng một số: lấy chữ số thấp nhất: bắt buộc `= n mod 2`; bỏ nó đi, phần còn lại biểu diễn `(n - r) / (-2)`; quy nạp cho thấy các chữ số trùng nhau. (Máy cũng kiểm tra tính duy nhất trong bảng Cách 1: `assert v not in table`.)

**Cách 3.** `divmod(n, -2) = (q, r')` với `r' in {0, -1}` và `n = -2 q + r'`. Nếu `r' = -1` thì `n = -2 q - 1 = -2 (q + 1) + 1`: chữ số `1`, thương `q + 1`. Đúng như hiệu chỉnh.

**Cách 4 (mẹo bit).** Gọi `X` là số nguyên thường có **cùng chuỗi bit** với biểu diễn cơ số `-2` của `n` (tức `X = sum b_i 2^i`). Tách `X = E + O`, với `E = sum_{i chẵn} b_i 2^i` và `O = sum_{i lẻ} b_i 2^i`. Theo định nghĩa cơ số `-2`: `n = E - O`. Mặt nạ `M` có bit `1` đúng ở các vị trí lẻ; `X xor M` **lật** các bit ở vị trí lẻ (bit `b` thành `1 - b`, đóng góp `2^i - b_i 2^i`) và giữ nguyên bit ở vị trí chẵn, nên `X xor M = E + (M - O) = n + M`. Vì phép XOR là **đối hợp** (`(A xor M) xor M = A`), suy ra `X = (n + M) xor M`. Đây đúng bất cứ khi nào biểu diễn nằm gọn trong số bit của `M` (64 bit, quá đủ cho `|n| <= 10^9`). Ta vẫn kiểm bằng máy bên dưới.

**Kiểm chứng bằng máy.** Oracle (Cách 1) liệt kê mọi chuỗi độ dài `<= 16` và đồng thời kiểm tính duy nhất; so với ba cách trên mọi `n <= 21845` (miền bảng phủ đủ), rồi kiểm giải mã cho `n` ngẫu nhiên tới `10^9`.

In [17]:
covered = [n for n in range(0, 21846)]
assert all(n in table for n in covered)
for n in covered:
    ref = table[n]
    assert base_neg2_loop(n) == base_neg2_divmod(n) == base_neg2_mask(n) == sol.baseNeg2(n) == sol_divmod(n) == sol_mask(n) == ref, n
print("mọi n = 0..21845: ba cách khớp oracle bảng (và biểu diễn duy nhất)")

rng = random.Random(1017)
for _ in range(20000):
    n = rng.randint(0, 10 ** 9)
    s = sol.baseNeg2(n)
    assert decode(s) == n and (s == "0" or s[0] == "1")
    assert s == base_neg2_divmod(n) == base_neg2_mask(n)
print("20000 số ngẫu nhiên tới 10^9: giải mã đúng, không có 0 đứng đầu, ba cách trùng nhau")

mọi n = 0..21845: ba cách khớp oracle bảng (và biểu diễn duy nhất)


20000 số ngẫu nhiên tới 10^9: giải mã đúng, không có 0 đứng đầu, ba cách trùng nhau


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Bảng mọi chuỗi ngắn | `O(2^L)` (chỉ phủ `n` nhỏ) | `O(2^L)` |
| 2. `n & 1` và chia | `O(log n)` | `O(log n)` |
| 3. `divmod` + hiệu chỉnh | `O(log n)` | `O(log n)` |
| 4. Mẹo bit | `O(1)` + in chuỗi | `O(log n)` |

Đo ở cỡ tối đa (`n = 10^9`) lặp nhiều lần:

In [18]:
n = 10 ** 9
for name, fn, reps in (("Cách 2", sol_loop, 20000), ("Cách 3", sol_divmod, 20000), ("Cách 4", sol_mask, 200000), ("nộp", sol.baseNeg2, 20000)):
    t0 = time.perf_counter()
    for _ in range(reps):
        got = fn(n)
    dt = (time.perf_counter() - t0) / reps * 1e6
    print(f"{name:7}: {dt:8.3f} µs/lần, baseNeg2(10^9) = {got} (độ dài {len(got)})")
assert decode(got) == n

Cách 2 :    6.060 µs/lần, baseNeg2(10^9) = 1001100111011111101111000000000 (độ dài 31)


Cách 3 :    4.309 µs/lần, baseNeg2(10^9) = 1001100111011111101111000000000 (độ dài 31)
Cách 4 :    0.234 µs/lần, baseNeg2(10^9) = 1001100111011111101111000000000 (độ dài 31)


nộp    :    4.693 µs/lần, baseNeg2(10^9) = 1001100111011111101111000000000 (độ dài 31)


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất. Vì biểu diễn **duy nhất**, ta so chuỗi **chính xác**.

In [19]:
def all_solutions():
    return {"loop": base_neg2_loop, "divmod": base_neg2_divmod, "mask": base_neg2_mask,
            "sol_loop": sol_loop, "sol_divmod": sol_divmod, "sol_mask": sol_mask, "final": sol.baseNeg2}

def check_all(n, expected=None):
    results = {name: fn(n) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (n, results)
    s = results["loop"]
    assert decode(s) == n and (s == "0" or s[0] == "1"), (n, s)
    if n in table:
        assert table[n] == s, (n, s, table[n])
    if expected is not None:
        assert s == expected, (n, s, expected)
    return s

# 12.1 ví dụ đề
for n, expected in examples:
    check_all(n, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (0, "0"),                      # trường hợp riêng: không có chữ số nào khác 0
    (1, "1"),                      # (-2)^0
    (5, "101"),                    # 4 + 1
    (6, "11010"),                  # 16 - 8 - 2
    (7, "11011"),                  # 16 - 8 - 2 + 1
    (8, "11000"),                  # 16 - 8
    (9, "11001"),                  # 16 - 8 + 1
    (10, "11110"),                 # 16 - 8 + 4 - 2
    (11, "11111"),                 # 16 - 8 + 4 - 2 + 1
    (12, "11100"),                 # 16 - 8 + 4
    (13, "11101"),                 # 16 - 8 + 4 + 1
    (16, "10000"),                 # (-2)^4
    (21845, "10" * 7 + "1"),       # 1 + 4 + 16 + ... + 4^7 = 21845: các bit 1 ở mọi vị trí chẵn từ 0 đến 14
]
for n, expected in edge:
    check_all(n, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm, so với oracle bảng)
def wrong_floor_division(n):                      # n //= -2 mà không trừ số dư trước
    if n == 0:
        return "0"
    digits = []
    while n:
        digits.append(str(n & 1))
        n //= -2
        if len(digits) > 80:                      # chặn vòng lặp vô hạn
            return "".join(reversed(digits))
    return "".join(reversed(digits))

def wrong_no_zero_case(n):                        # quên n == 0: trả chuỗi rỗng
    digits = []
    while n:
        r = n & 1
        digits.append(str(r))
        n = -((n - r) // 2)
    return "".join(reversed(digits))

def wrong_no_reverse(n):                          # quên đảo chuỗi
    if n == 0:
        return "0"
    digits = []
    while n:
        r = n & 1
        digits.append(str(r))
        n = -((n - r) // 2)
    return "".join(digits)

def wrong_divmod_no_carry(n):                     # hiệu chỉnh số dư nhưng quên cộng 1 vào thương
    if n == 0:
        return "0"
    digits = []
    while n:
        n, r = divmod(n, -2)
        if r < 0:
            r += 2
        digits.append(str(r))
        if len(digits) > 80:
            break
    return "".join(reversed(digits))

def wrong_plain_binary(n):                        # dùng cơ số 2 thông thường
    return bin(n)[2:]

def wrong_leading_zeros(n):                       # độ dài cố định 32 chữ số (có số 0 đứng đầu)
    return base_neg2_loop(n).zfill(32)

def find_counter(wrong, limit=100):
    for n in range(0, limit + 1):
        if wrong(n) != table[n]:
            return n
    return None

counters = {}
for name, fn in (("n //= -2 không trừ số dư", wrong_floor_division), ("quên trường hợp n = 0", wrong_no_zero_case),
                 ("quên đảo chuỗi", wrong_no_reverse), ("hiệu chỉnh quên cộng 1 vào thương", wrong_divmod_no_carry),
                 ("dùng cơ số 2 thường", wrong_plain_binary), ("thừa số 0 đứng đầu", wrong_leading_zeros)):
    n = find_counter(fn)
    assert n is not None, name
    counters[name] = n
    print(f"{name:36} n={n}: sai -> {fn(n)!r}, đúng -> {table[n]!r}")
    check_all(n)
print("12.3 phản ví dụ: OK")

n //= -2 không trừ số dư             n=1: sai -> '11', đúng -> '1'
quên trường hợp n = 0                n=0: sai -> '', đúng -> '0'
quên đảo chuỗi                       n=2: sai -> '011', đúng -> '110'
hiệu chỉnh quên cộng 1 vào thương    n=1: sai -> '11', đúng -> '1'
dùng cơ số 2 thường                  n=2: sai -> '10', đúng -> '110'
thừa số 0 đứng đầu                   n=0: sai -> '00000000000000000000000000000000', đúng -> '0'
12.3 phản ví dụ: OK


In [21]:
# 12.4 toàn miền bảng đã kiểm ở mục 10; ở đây: ngẫu nhiên tới 10^9, biên, và các lũy thừa của 2 / 4 lân cận
rng = random.Random(4017)
special = [0, 1, 10 ** 9, 10 ** 9 - 1, 2 ** 29, 2 ** 30 - 1, 4 ** 14, 4 ** 14 + 1, 4 ** 14 - 1, 2 * 4 ** 14]
for n in special:
    check_all(n)
for _ in range(3000):
    check_all(rng.randint(0, 10 ** 9))
print("12.4 biên và ngẫu nhiên (3000 số): OK")

# 12.5 tính chất: độ dài, chẵn lẻ, nhân hai
for n in range(0, 5000):
    s = sol.baseNeg2(n)
    assert (s[-1] == "1") == (n % 2 == 1)                    # chữ số cuối là tính chẵn lẻ
    assert len(s) <= n.bit_length() + 2 or n == 0            # dài hơn nhị phân thường tối đa 2 chữ số
    assert sol.baseNeg2(n) == "0" or s.count("1") >= 1
    # (n) và -2n: thêm một chữ số 0 vào cuối đổi giá trị thành -2n
    assert decode(s + "0") == -2 * n
print("12.5 tính chất: OK")

12.4 biên và ngẫu nhiên (3000 số): OK
12.5 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `n //= -2` mà không trừ số dư trước

Với `n = 3`: chữ số `1`, nhưng `3 // -2 = -2` (chia sàn), trong khi cần `(3 - 1) / (-2) = -1`. Sai lệch tích lũy làm mọi chữ số sau đều sai.

In [22]:
n = counters["n //= -2 không trừ số dư"]
print(f"n={n}: sai ->", wrong_floor_division(n), "| đúng ->", Solution().baseNeg2(n))
assert wrong_floor_division(n) != Solution().baseNeg2(n)
print("3 // -2 =", 3 // -2, "(cần -1)")

n=1: sai -> 11 | đúng -> 1
3 // -2 = -2 (cần -1)


Vì sao sai: chia sàn làm tròn xuống `-vô cực`, không phải chia chính xác. **Đúng:** trừ chữ số thấp nhất `r` trước (khi đó `n - r` chẵn) rồi mới chia: `n = -((n - r) // 2)`.

### Lỗi 2 — Quên trường hợp `n = 0`

Vòng `while n` không chạy nên chuỗi rỗng; đề yêu cầu `"0"`.

In [23]:
n = counters["quên trường hợp n = 0"]
print(f"n={n}: sai ->", repr(wrong_no_zero_case(n)), "| đúng ->", repr(Solution().baseNeg2(n)))
assert wrong_no_zero_case(n) != Solution().baseNeg2(n)

n=0: sai -> '' | đúng -> '0'


Vì sao sai: `n = 0` không có chữ số nào được sinh. **Đúng:** `if n == 0: return "0"`.

### Lỗi 3 — Quên đảo chuỗi

Chữ số được sinh từ **thấp lên cao**; ghép nguyên thứ tự cho chuỗi ngược.

In [24]:
n = counters["quên đảo chuỗi"]
print(f"n={n}: sai ->", repr(wrong_no_reverse(n)), "| đúng ->", repr(Solution().baseNeg2(n)))
assert wrong_no_reverse(n) != Solution().baseNeg2(n)

n=2: sai -> '011' | đúng -> '110'


Vì sao sai: chữ số đầu tiên sinh ra là chữ số hàng đơn vị (bên phải). **Đúng:** `"".join(reversed(digits))`.

### Lỗi 4 — Hiệu chỉnh số dư âm mà quên cộng `1` vào thương

Với `divmod(3, -2) = (-2, -1)`, chữ số `1` nhưng thương phải là `-1` (không phải `-2`).

In [25]:
n = counters["hiệu chỉnh quên cộng 1 vào thương"]
print(f"n={n}: sai ->", wrong_divmod_no_carry(n), "| đúng ->", Solution().baseNeg2(n))
assert wrong_divmod_no_carry(n) != Solution().baseNeg2(n)

n=1: sai -> 11 | đúng -> 1


Vì sao sai: `n = -2 * q + (-1) = -2 * (q + 1) + 1`; đổi số dư `-1` thành `1` đòi tăng thương thêm `1`. **Đúng:** `r += 2; q += 1` cùng lúc.

### Lỗi 5 — Dùng cơ số 2 thường, hoặc thừa số `0` đứng đầu

`bin(n)[2:]` là cơ số `2`, và việc đệm cho đủ độ dài cố định tạo số `0` đứng đầu (đề cấm).

In [26]:
n = counters["dùng cơ số 2 thường"]
print(f"n={n}: sai ->", wrong_plain_binary(n), "| đúng ->", Solution().baseNeg2(n))
assert wrong_plain_binary(n) != Solution().baseNeg2(n)
n = counters["thừa số 0 đứng đầu"]
print(f"n={n}: sai ->", wrong_leading_zeros(n), "| đúng ->", Solution().baseNeg2(n))
assert wrong_leading_zeros(n) != Solution().baseNeg2(n)

n=2: sai -> 10 | đúng -> 110
n=0: sai -> 00000000000000000000000000000000 | đúng -> 0


Vì sao sai: `bin(6) = "110"` giải mã trong cơ số `-2` thành `2`, không phải `6`. **Đúng:** trọng số `(-2)^i` và không đệm số `0`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đổi sang cơ số âm / cơ số lạ" ⇒ **chia liên tiếp** với số dư được ép về miền chữ số hợp lệ `{0, ..., |b| - 1}`.
- Quy tắc: `r = n mod |b|` (không âm), `q = (n - r) / b`; luôn **trừ số dư trước khi chia** để không dính chia sàn.
- Chữ số thấp nhất quyết định bởi `n & 1` đúng cả với số âm trong Python (`-3 & 1 == 1`).
- Mẹo bit đặc biệt cho cơ số `-2`: `(n + 0xAAAA...) ^ 0xAAAA...`.
- Kiểm tra độc lập: **giải mã** chuỗi bằng `sum d_i * (-2)^i` và so với `n`.
- Cùng họ: Add Two Numbers in base -2 (1073), Convert a Number to Hexadecimal (405), Excel Sheet Column Title (168).
- Test nên có: `n = 0`, `n = 1`, các lũy thừa của `2` và `4`, `n = 10^9`.

## 15. Biến thể

1. **Số âm làm đầu vào:** thuật toán không đổi (đúng cho mọi số nguyên).
2. **Cơ số âm tổng quát `-b`:** chữ số `0..b-1` (bài tập 5).
3. **Cộng hai số trong cơ số `-2`** (bài 1073).
4. Bài liên quan: 1073, 405, 168.

Biến thể 1: cho `n` âm, kiểm bằng giải mã (và so với bảng Cách 1):

In [27]:
for n in range(-300, 301):
    s = base_neg2_loop(n)
    assert decode(s) == n and (s == "0" or s[0] == "1"), (n, s)
    assert s == base_neg2_divmod(n)
    if n in table:
        assert table[n] == s
print("số âm: -3 ->", base_neg2_loop(-3), "| -1 ->", base_neg2_loop(-1), "| -10 ->", base_neg2_loop(-10))
M32 = 0xAAAAAAAAAAAAAAAA
assert all(bin((n + M32) ^ M32)[2:] == base_neg2_loop(n) for n in range(0, 1000))
print("biến thể số âm: khớp giải mã và bảng Cách 1")

số âm: -3 -> 1101 | -1 -> 11 | -10 -> 1010
biến thể số âm: khớp giải mã và bảng Cách 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bảng mọi chuỗi ngắn | Liệt kê và giải mã | `O(2^L)` | `O(2^L)` | Oracle, kiểm tính duy nhất |
| 2. `n & 1` và chia | Chữ số thấp nhất = chẵn lẻ | `O(log n)` | `O(log n)` | **Cách nộp chính** |
| 3. `divmod` + hiệu chỉnh | Số dư `{0, -1}` đưa về `{0, 1}` | `O(log n)` | `O(log n)` | Khi muốn dùng `divmod` |
| 4. Mẹo bit | `(n + M) ^ M` | `O(1)` | `O(log n)` | Gọn nhất, chỉ cơ số `-2` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(divmod(3, -2), divmod(2, -2), divmod(-1, -2))
print(3 & 1, -3 & 1, -4 & 1, decode("110"))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ba cặp
my_guess_2 = None      # bốn giá trị

### Bài 2 — Tính tay (dễ-vừa)

Tính tay biểu diễn cơ số `-2` của `n = 7` (viết chuỗi bit), rồi kiểm tra bằng cách cộng các trọng số `(-2)^i`.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành vòng chia.

In [30]:
def exercise_3(n: int) -> str:
    if n == 0:
        return "0"
    digits = []
    while n:
        # TODO 1: r = chữ số thấp nhất (tính chẵn lẻ của n)
        # TODO 2: thêm r (dạng chuỗi) vào digits
        # TODO 3: cập nhật n = -((n - r) // 2)
        pass
    # TODO 4: đảo digits và ghép
    pass


# for n, expected in examples:
#     assert exercise_3(n) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(n)` bằng **mẹo bit** `(n + M) ^ M`.

In [31]:
def exercise_4(n: int) -> str:
    # TODO
    pass


# for n, expected in examples:
#     assert exercise_4(n) == expected

### Bài 5 — Mini challenge: cơ số âm tổng quát (khó)

Viết `exercise_5(n, base)` với `base` là số nguyên **âm** `<= -2`, trả về chuỗi các chữ số (mỗi chữ số trong `0..|base|-1`, với `|base| <= 10` nên mỗi chữ số một ký tự) biểu diễn `n` trong cơ số `base`, không có `0` đứng đầu (trừ `"0"`).

```python
exercise_5(2, -2)   -> "110"
exercise_5(4, -3)   -> "121"      # 9 + 2 * (-3) + 1 = 4
exercise_5(0, -5)   -> "0"
exercise_5(10, -10) -> "190"      # 100 + 9 * (-10) + 0 = 10
```
Gợi ý: `r = n % |base|` (không âm, đúng với `n` âm trong Python), `n = (n - r) // base`.

In [32]:
def exercise_5(n: int, base: int) -> str:
    # TODO
    pass


# assert exercise_5(2, -2) == "110"
# assert exercise_5(4, -3) == "121"
# assert exercise_5(0, -5) == "0"
# assert exercise_5(10, -10) == "190"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `divmod(a, b)` với `b < 0` cho số dư cùng dấu với `b` (không dương).
- Mức 2: `divmod(3, -2) = (-2, -1)`; `divmod(2, -2) = (-1, 0)`; `divmod(-1, -2) = (0, -1)`.
- Mức 3: dòng 1: `(-2, -1) (-1, 0) (0, -1)`; dòng 2: `1 1 0 2`.

**Bài 2**
- Mức 1: `7` lẻ nên chữ số cuối `1`; `q = -((7 - 1) // 2) = -3`.
- Mức 2: `-3` lẻ nên chữ số `1`, `q = -((-3 - 1) // 2) = 2`; rồi `2`: chữ số `0`, `q = -1`; `-1`: chữ số `1`, `q = 1`; `1`: chữ số `1`, `q = 0`.
- Mức 3: các chữ số từ thấp lên cao `1, 1, 0, 1, 1`; chuỗi `"11011"` (`16 - 8 - 2 + 1 = 7`).

**Bài 3**
- Mức 1: `r = n & 1`.
- Mức 2: `digits.append(str(r))`.
- Mức 3: `n = -((n - r) // 2)`; `return "".join(reversed(digits))`.

**Bài 4**
- Mức 1: `M = 0xAAAAAAAAAAAAAAAA`.
- Mức 2: `v = (n + M) ^ M`.
- Mức 3: `return bin(v)[2:]` (với `n = 0` cho `"0"`).

**Bài 5**
- Mức 1: `b = -base` (dương); `r = n % b`.
- Mức 2: `n = (n - r) // base` (chia cho số âm; `n - r` chia hết cho `b`).
- Mức 3: ghép `str(r)` theo thứ tự đảo; nhớ `n = 0` trả `"0"`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(divmod(3, -2), divmod(2, -2), divmod(-1, -2))
print(3 & 1, -3 & 1, -4 & 1, decode("110"))

assert (divmod(3, -2), divmod(2, -2), divmod(-1, -2)) == ((-2, -1), (-1, 0), (0, -1))
assert (3 & 1, -3 & 1, -4 & 1, decode("110")) == (1, 1, 0, 2)

(-2, -1) (-1, 0) (0, -1)
1 1 0 2


### Đáp án Bài 2

`7`: chữ số `1`, thương `-3`; `-3`: chữ số `1`, thương `2`; `2`: chữ số `0`, thương `-1`; `-1`: chữ số `1`, thương `1`; `1`: chữ số `1`, thương `0`. Đảo: `"11011"`. Kiểm tra: `16 - 8 + 0 - 2 + 1 = 7`.

In [34]:
hand_answer_solution = "11011"
assert Solution().baseNeg2(7) == hand_answer_solution
assert decode(hand_answer_solution) == 7 == 16 - 8 + 0 - 2 + 1
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(n: int) -> str:
    if n == 0:
        return "0"
    digits = []
    while n:
        r = n & 1
        digits.append(str(r))
        n = -((n - r) // 2)
    return "".join(reversed(digits))

for n, expected in examples:
    assert exercise_3_solution(n) == expected
for n in range(0, 2000):
    assert exercise_3_solution(n) == table[n]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(n: int) -> str:
    M = 0xAAAAAAAAAAAAAAAA
    return bin((n + M) ^ M)[2:]

for n, expected in examples:
    assert exercise_4_solution(n) == expected
for n in range(0, 2000):
    assert exercise_4_solution(n) == table[n]
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(n: int, base: int) -> str:
    b = -base
    if n == 0:
        return "0"
    digits = []
    while n:
        r = n % b                          # 0 <= r < b, đúng cả với n âm trong Python
        digits.append(str(r))
        n = (n - r) // base                # (n - r) chia hết cho b
    return "".join(reversed(digits))

def decode_base(s: str, base: int) -> int:
    return sum(int(c) * base ** i for i, c in enumerate(reversed(s)))

assert exercise_5_solution(2, -2) == "110"
assert exercise_5_solution(4, -3) == "121"
assert exercise_5_solution(0, -5) == "0"
assert exercise_5_solution(10, -10) == "190"
rng = random.Random(5)
for _ in range(2000):
    base = -rng.randint(2, 10)
    n = rng.randint(-10 ** 6, 10 ** 6)
    s = exercise_5_solution(n, base)
    assert decode_base(s, base) == n and (s == "0" or s[0] != "0"), (n, base, s)
for n in range(0, 500):
    assert exercise_5_solution(n, -2) == table[n]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (chia liên tiếp cho `-2`)

```python
class Solution:
    def baseNeg2(self, n: int) -> str:
        if n == 0:
            return "0"
        digits = []
        while n:
            r = n & 1                      # chữ số thấp nhất = tính chẵn lẻ của n
            digits.append(str(r))
            n = -((n - r) // 2)            # trừ chữ số rồi chia cho -2
        return "".join(reversed(digits))   # đảo vì sinh từ thấp lên cao
```

Công thức nhớ nhanh: `r = n & 1`; `n = -((n - r) // 2)`; hoặc mẹo bit `bin((n + 0xAAAA...) ^ 0xAAAA...)`.

### Checklist trước khi submit

- [ ] Xử lý riêng `n == 0` (trả `"0"`)?
- [ ] Trừ chữ số thấp nhất `r` **trước** khi chia (không dùng `n //= -2` trần)?
- [ ] Đảo chuỗi chữ số (sinh từ thấp lên cao)?
- [ ] Không có số `0` đứng đầu?
- [ ] Nếu dùng `divmod(n, -2)`: hiệu chỉnh `r == -1` thành `r = 1` và `n += 1`?
- [ ] Đã giải mã lại bằng `sum d_i * (-2)^i` để kiểm tra, và thử `n = 0, 1, 2, 3, 4, 10^9`?